# FMC Co-Retrieval from MODIS + GEDI Tree Height using Earth Engine and PyTorch MLP

**Description:** This notebook retrieves **Fuel Moisture Content (FMC)** by jointly inverting
Equivalent Water Thickness (EWT) and Dry Matter Content (Cm) from MODIS MCD43A4 reflectance
data, with LAI from MCD15A3H, land-cover type from MCD12Q1, and **static tree height from
GEDI V2.7** for woody vegetation types (Broadleaf, Conifer, Shrub).

**Method overview:**
1. Build a daily MODIS reflectance + LAI stack with quality control.
2. Attach static GEDI canopy height as a fixed input band for woody vegetation.
3. Compute spectral water index (SWI) and normalised spectral angle index (NSAI).
4. Run a pre-trained 4-layer MLP (tanh activation) → EWT, Cm → $FMC = EWT / Cm \times 100$.
5. Use vegetation-type-specific model weights and normalisation parameters.

**Key differences from the baseline version:**
- **Tree height** from GEDI L4A V2.7 (`users/potapovpeter/GEDI_V27`) is added as a fixed
  input for Broadleaf, Conifer, and Shrub. Grass does **not** use tree height.
- The model variant (with-LAI vs without-LAI) is still chosen **solely** based on LAI
  quality; tree height is always included for woody types regardless of quality level.
- With-LAI woody models have **11 inputs** (7 reflectance + LAI + NSAI + SWI + TreeHeight).
- Without-LAI woody models have **10 inputs** (7 reflectance + NSAI + SWI + TreeHeight).

**Requirements:**
- Google Earth Engine Python API (`ee`) authenticated with a GEE project.
- PyTorch (`torch`) for loading model weights.
- Pre-trained `.pkl` model files placed in the working directory.
- Access to the `users/potapovpeter/GEDI_V27` asset (or equivalent GEDI canopy height).

**Authors / Citation:** *(add your info here)*

**License:** *(add your license here, e.g. MIT)*

## 0. Configuration

Edit these values before running.

In [ ]:
# ============================================================
#  User-configurable parameters — change these for your project
# ============================================================

GEE_PROJECT_ID = "ee-ahu1293659-10"          # Your Google Earth Engine project ID
GEE_ASSET_FOLDER = f"projects/{GEE_PROJECT_ID}/assets"  # Asset export destination

# Date range for retrieval
DATA_START = "2019-01-01"
DATA_END   = "2019-08-01"

# MCD12Q1 land-cover year (usually the year of DATA_START)
LC_YEAR = 2019

# Model file directory (set to the folder containing your .pkl files)
MODEL_DIR = "."

# Export parameters (used in the final section)
EXPORT_START_DATE = "2019-01-01"
EXPORT_END_DATE   = "2019-01-02"
EXPORT_SCALE_M    = 500          # metres per pixel
EXPORT_MAX_PIXELS = 1e13

# Scale factor constants
REFLECTANCE_SCALE = 0.0001       # MCD43A4 reflectance scale factor
LAI_SCALE         = 0.1          # MCD15A3H LAI scale factor
FMC_SCALE         = 100          # Scale FMC to integer percentage (0–400)

# ---- Tree Height Configuration ----
# Pre-processed static canopy height (500 m, EPSG:4326, uint8, 0–60).
# ⚠️  This is a DERIVED product — see DATA.md for the original data source.
# Users MUST access and cite the original GEDI L4A V2.7 data, not just this asset.
TREE_HEIGHT_ASSET = "projects/ee-ahu1293659-11/assets/forest_height_2019_global"
# ---------------------------------------

## 1. Environment Setup & Imports

In [2]:
import os
import sys
from datetime import datetime, timedelta

import ee
import geemap
import numpy as np
import torch
import torch.nn as nn

# -----------------------------------------------------------------
# Clean up proxy environment variables that may interfere with
# Earth Engine authentication in some network environments (e.g. Colab).
# -----------------------------------------------------------------
for var in ["HTTP_PROXY", "HTTPS_PROXY", "http_proxy", "https_proxy"]:
    os.environ.pop(var, None)
print("Proxy environment variables cleared.")

Proxy environment variables cleared.


### 1.1 Authenticate & Initialise Earth Engine

The authentication method depends on your environment:
- **Google Colab:** Uses `google.colab.auth` (interactive browser login).
- **Local / Vertex AI / other:** Uses `ee.Authenticate()` or existing credentials.

If you are NOT running in Colab, comment out the `auth.authenticate_user()` line
and make sure you have run `ee.Authenticate()` once beforehand.

In [3]:
# ----- Authenticate (Colab: interactive; local: reuse saved credentials) -----
try:
    from google.colab import auth
    auth.authenticate_user()
    print("Colab authentication completed.")
except ImportError:
    # Not running in Colab — assume ee.Authenticate() has been run previously
    print("Running outside Colab; using existing Earth Engine credentials.")

# ----- Initialise Earth Engine -----
ee.Initialize(project=GEE_PROJECT_ID)
print(f"Earth Engine initialised with project '{GEE_PROJECT_ID}'.")
print("Test:", ee.Number(1).getInfo())

Colab authentication completed.
Earth Engine initialised with project 'ee-ahu1293659-10'.
Test: 1


### 1.2 Register PyTorch Classes for Safe Deserialisation

Needed to load the pre-trained `.pkl` models with `torch.load()`.
The `MLP` class accepts an `input_dim` parameter to support different numbers
of input features across vegetation types (10 for Grass, 11 for woody with tree height).

In [4]:
# -----------------------------------------------------------------
# Define the MLP architecture — supports BOTH conventions:
#
#   Layout A (fc1–fc4):  state_dict has fc1.weight, fc1.bias, …
#   Layout B (Sequential): state_dict has net.0.weight, net.0.bias, …
#
# The notebook auto-detects which layout a saved model uses and
# builds the matching architecture during load_model().
# -----------------------------------------------------------------
class MLP(nn.Module):
    """4-layer MLP with tanh activation — two possible internal layouts.

    Architecture (identical computation):
        Linear(input_dim → 64) → tanh
        Linear(64 → 128)       → tanh
        Linear(128 → 64)       → tanh
        Linear(64 → 2)          ← output, NO activation
    """

    def __init__(self, input_dim=10, use_sequential=False):
        super(MLP, self).__init__()
        self.input_dim = input_dim
        self.use_sequential = use_sequential

        if use_sequential:
            # Layout B — net.0, net.2, net.4, net.6
            self.net = nn.Sequential(
                nn.Linear(input_dim, 64),   # net.0
                nn.Tanh(),                   # net.1
                nn.Linear(64, 128),          # net.2
                nn.Tanh(),                   # net.3
                nn.Linear(128, 64),          # net.4
                nn.Tanh(),                   # net.5
                nn.Linear(64, 2),            # net.6
            )
        else:
            # Layout A — fc1, fc2, fc3, fc4
            self.fc1 = nn.Linear(input_dim, 64)
            self.fc2 = nn.Linear(64, 128)
            self.fc3 = nn.Linear(128, 64)
            self.fc4 = nn.Linear(64, 2)

    def forward(self, x):
        if self.use_sequential:
            return self.net(x)
        x = torch.tanh(self.fc1(x))
        x = torch.tanh(self.fc2(x))
        x = torch.tanh(self.fc3(x))
        x = self.fc4(x)
        return x


print("MLP class defined (auto-detects fc1 vs Sequential layout).")

MLP class defined (auto-detects fc1 vs Sequential layout).


## 2. Data Loading — MODIS Collections & GEDI Tree Height

In [ ]:
# -----------------------------------------------------------------
# Load MODIS image collections
#   MCD43A4  — NBAR surface reflectance (500 m daily)
#   MCD15A3H — LAI / FPAR (500 m, 4-day composite)
#   MCD12Q1  — Land-cover type (500 m annual)
# -----------------------------------------------------------------
mcd43a4  = ee.ImageCollection("MODIS/061/MCD43A4").filterDate(DATA_START, DATA_END)
mcd15a3h = ee.ImageCollection("MODIS/061/MCD15A3H").filterDate(DATA_START, DATA_END)
mcd12q1  = ee.Image(f"MODIS/061/MCD12Q1/{LC_YEAR}_01_01")

print(f"MCD43A4  images: {mcd43a4.size().getInfo()}")
print(f"MCD15A3H images: {mcd15a3h.size().getInfo()}")

# -----------------------------------------------------------------
# Load pre-processed static canopy height (2019, 500 m, EPSG:4326)
#
# This asset was derived from GEDI L4A V2.7 by upscaling the native
# ~0.00025° resolution to 500 m and masking values > 60 m.
# See DATA.md for the original data source and processing steps.
#
# Asset: projects/ee-ahu1293659-11/assets/forest_height_2019_global
#   Band:   b1 (unsigned int8, 0–60)
#   CRS:    EPSG:4326
#   Scale:  ~500 m
#
# ⚠️  IMPORTANT — this is a pre-processed DERIVED product.
#     Users MUST obtain and cite the original GEDI L4A V2.7 data
#     (Potapov et al.) rather than relying solely on this asset.
#     See DATA.md for details.
# -----------------------------------------------------------------
tree_height_img = ee.Image(TREE_HEIGHT_ASSET).rename("TreeHeight")

print(f"Tree height loaded from: {TREE_HEIGHT_ASSET}")
print(f"  Band b1 → renamed to 'TreeHeight'")
print(f"  Already at 500 m EPSG:4326 — no reprojection needed")

### 2.1 IGBP Land-Cover Reclassification

Map the 17-class IGBP legend to 4 broad vegetation types:

| Value | Vegetation Type | Original IGBP Classes | Uses Tree Height? |
|-------|-----------------|----------------------|-------------------|
| 1     | Grass           | 8, 9, 10, 12, 14     | No |
| 2     | Broadleaf Forest| 2, 4, 5              | **Yes** |
| 3     | Conifer Forest  | 1, 3                 | **Yes** |
| 4     | Shrub           | 6, 7                 | **Yes** |
| 255   | Other (masked)  | all others           | — |

In [6]:
igbp = mcd12q1.select("LC_Type1")

# Reclassify IGBP → 4 broad vegetation types; 255 = non-vegetated / water
reclassified = igbp.expression(
    "(b('LC_Type1') == 8  || b('LC_Type1') == 9  ||"
    " b('LC_Type1') == 10 || b('LC_Type1') == 12 ||"
    " b('LC_Type1') == 14) ? 1"
    ": (b('LC_Type1') == 2 || b('LC_Type1') == 4 ||"
    " b('LC_Type1') == 5)  ? 2"
    ": (b('LC_Type1') == 1 || b('LC_Type1') == 3)  ? 3"
    ": (b('LC_Type1') == 6 || b('LC_Type1') == 7)  ? 4"
    ": 255"
).rename("reclassified")

# Mask out non-vegetated pixels (255)
reclassified = reclassified.updateMask(reclassified.neq(255))
print("Land-cover reclassification complete.")

Land-cover reclassification complete.


## 3. Image Preprocessing Functions

In [7]:
# -----------------------------------------------------------------
#  3.1  Attach closest-in-time LAI to each MCD43A4 image
# -----------------------------------------------------------------
def add_lai_to_mcd43a4(image):
    """For each MCD43A4 image, find the temporally closest MCD15A3H LAI image
    within a ±4 day window and attach its Lai, FparLai_QC, FparExtra_QC bands.

    Pixels with LAI == 0 are masked (as per the paper's methodology).
    """
    date = ee.Date(image.get("system:time_start"))

    # Search window: [-4, +5) days around the reflectance image date
    lai_images = mcd15a3h.filterDate(
        date.advance(-4, "day"), date.advance(5, "day")
    )

    # Find the LAI image closest in time
    def _compute_difference(lai_img):
        lai_date = ee.Date(lai_img.get("system:time_start"))
        diff = date.difference(lai_date, "day").abs()
        return lai_img.set("date_difference", diff)

    lai_images = lai_images.map(_compute_difference)
    min_diff = lai_images.aggregate_min("date_difference")
    closest = lai_images.filter(ee.Filter.eq("date_difference", min_diff))
    lai_image = ee.ImageCollection(closest).first()

    # Select bands and match projection
    lai_image = lai_image.select(
        ["Lai", "FparLai_QC", "FparExtra_QC"]
    ).setDefaultProjection(image.projection())

    # Mask out LAI == 0 (required per paper methodology)
    lai_image = lai_image.updateMask(lai_image.select("Lai").gt(0))

    return image.addBands(lai_image)

In [8]:
# -----------------------------------------------------------------
#  3.2  Scale reflectance and LAI to physical units
# -----------------------------------------------------------------
def process_image(image):
    """Scale MCD43A4 reflectance bands (×0.0001) and LAI (×0.1) to physical values.

    Returns an image with:
        Nadir_Reflectance_Band1–7   (scaled reflectance)
        Lai                          (scaled LAI)
        FparLai_QC, FparExtra_QC    (LAI quality flags)
        BRDF_Albedo_Band_Mandatory_Quality_Band1–7  (reflectance quality flags)
    """
    REFLECTANCE_BANDS = ["1", "2", "3", "4", "5", "6", "7"]
    QUALITY_BANDS = [
        f"BRDF_Albedo_Band_Mandatory_Quality_Band{i}" for i in range(1, 8)
    ]

    processed = ee.Image()

    # Scale reflectance bands
    for idx in REFLECTANCE_BANDS:
        band_name = f"Nadir_Reflectance_Band{idx}"
        scaled = image.select(band_name).multiply(REFLECTANCE_SCALE)
        processed = processed.addBands(scaled)

    # Scale LAI band
    lai_scaled = image.select("Lai").multiply(LAI_SCALE)
    processed = processed.addBands(lai_scaled)

    # Copy QC bands
    qc_bands = image.select(
        ["FparLai_QC", "FparExtra_QC"] + QUALITY_BANDS
    )
    processed = processed.addBands(qc_bands)

    return processed.copyProperties(image, ["system:time_start"])

In [9]:
# -----------------------------------------------------------------
#  3.3  Quality-control masking and quality-level band
# -----------------------------------------------------------------
def QCprocess_image(image):
    """Apply MODIS quality flags to build a 4-level quality band.

    Quality levels (priority: 0 > 1 > 2 > 3):
        0 — Both reflectance AND LAI pass QC (best quality).
        1 — Reflectance passes QC; LAI is not considered.
        2 — Both reflectance and LAI have valid pixels (no QC requirement).
        3 — Only reflectance has valid pixels (lowest quality).
    """
    REFLECTANCE_BANDS = ["1", "2", "3", "4", "5", "6", "7"]

    # ---- Build a mask: all 7 reflectance bands have valid data ----
    reflectance_masks = []
    for idx in REFLECTANCE_BANDS:
        band_mask = image.select(f"Nadir_Reflectance_Band{idx}").mask()
        band_mask = band_mask.updateMask(band_mask)
        reflectance_masks.append(band_mask)

    reflectance_mask = reflectance_masks[0]
    for m in reflectance_masks[1:]:
        reflectance_mask = reflectance_mask.And(m)

    # ---- LAI valid-data mask ----
    lai_mask = image.select("Lai").mask()
    lai_mask = lai_mask.updateMask(lai_mask)

    # ---- MCD43A4 quality mask (bit 0 == 0 → best quality) ----
    quality_mask_MCD43A4 = image.select(
        "BRDF_Albedo_Band_Mandatory_Quality_Band1"
    ).bitwiseAnd(1).eq(0)
    for i in range(2, 8):
        band_qc = f"BRDF_Albedo_Band_Mandatory_Quality_Band{i}"
        current = image.select(band_qc).bitwiseAnd(1).eq(0)
        quality_mask_MCD43A4 = quality_mask_MCD43A4.And(current)
    quality_mask_MCD43A4 = quality_mask_MCD43A4.updateMask(quality_mask_MCD43A4)

    # ---- LAI quality mask (MCD15A3H QC bits) ----
    fpar_lai_qc = image.select("FparLai_QC")
    modland_qc_mask = fpar_lai_qc.bitwiseAnd(1).eq(0)              # Bit 0
    cloud_state_mask = fpar_lai_qc.rightShift(3).bitwiseAnd(3).eq(0)  # Bits 3-4

    fpar_extra_qc = image.select("FparExtra_QC")
    snow_ice_mask = fpar_extra_qc.rightShift(2).bitwiseAnd(1).eq(0)   # Bit 2
    cloud_shadow_mask = fpar_extra_qc.rightShift(6).bitwiseAnd(1).eq(0)  # Bit 6

    quality_mask_LAI = (
        modland_qc_mask.And(cloud_state_mask)
        .And(snow_ice_mask)
        .And(cloud_shadow_mask)
    )
    quality_mask_LAI = quality_mask_LAI.updateMask(quality_mask_LAI)

    # ---- Build hierarchical quality levels ----
    quality_level_0 = quality_mask_MCD43A4.And(quality_mask_LAI)   # best
    quality_level_1 = quality_mask_MCD43A4                          # reflectance QC only
    quality_level_2 = reflectance_mask.And(lai_mask)                # both have values
    quality_level_3 = reflectance_mask                              # reflectance only

    # Stack with priority (0 wins over 1 over 2 over 3)
    quality_band = (
        quality_level_3.where(quality_level_3, 3)
        .where(quality_level_2, 2)
        .where(quality_level_1, 1)
        .where(quality_level_0, 0)
    )

    return image.addBands(quality_band.rename("Quality_Level"))

In [ ]:
# -----------------------------------------------------------------
#  3.4  Attach static tree height to each image
# -----------------------------------------------------------------
def add_tree_height(image):
    """Attach the pre-processed static canopy height band to the image.

    Tree height is a fixed input for woody vegetation types (Broadleaf,
    Conifer, Shrub). It is NOT used by the Grass model but is present
    on the image for all pixels.

    The asset is already at 500 m in EPSG:4326 — GEE handles the
    on-the-fly reprojection to MODIS sinusoidal automatically.
    """
    return image.addBands(tree_height_img)

## 4. Spectral Indices & Model Inference

In [12]:
# -----------------------------------------------------------------
#  4.1  Spectral Water Index (SWI)
# -----------------------------------------------------------------
def SACEWT(image):
    """Compute the Spectral Absorption Cosine of Equivalent Water Thickness (SWI)
    from bands 2 (859 nm) and 5 (1240 nm).

    Reference: Wang et al. (2020) — SWI = cos(θ) between the reflectance vector
    and the water absorption coefficient vector at 859 & 1240 nm.
    """
    # Central wavelengths (nm) and water absorption coefficients (cm⁻¹)
    all_bands = np.array([
        412, 443, 469, 488, 531, 551, 555, 645, 667, 678,
        748, 859, 869, 905, 936, 940, 1240, 1375, 1640, 2130
    ])
    all_absorbing = np.array([
        0.000202945, 0.000208502, 0.000131038, 0.00022918,
        0.000497591, 0.000594837, 0.00062043,  0.003341514,
        0.005383188, 0.005862709, 0.026663944, 0.043961961,
        0.047051415, 0.069649995, 0.157152966, 0.197166888,
        1.146759147, 9.053919179, 6.154778038, 25.68018038
    ])

    # Select absorption coefficients at 859 nm and 1240 nm
    midband = [859, 1240]
    indices = [np.where(all_bands == v)[0][0] for v in midband]
    a1, a2 = all_absorbing[indices]

    band2 = image.select("Nadir_Reflectance_Band2")   # 859 nm
    band5 = image.select("Nadir_Reflectance_Band5")   # 1240 nm

    # SWI = (r₂·a₂ + r₅·a₅) / (||r|| · ||a||)
    result = image.expression(
        "numerator / denominator",
        {
            "numerator": image.expression(
                "r2 * a1 + r5 * a2",
                {"r2": band2, "r5": band5, "a1": a1, "a2": a2}
            ),
            "denominator": image.expression(
                "sqrt(r2 * r2 + r5 * r5) * sqrt(a1 * a1 + a2 * a2)",
                {"r2": band2, "r5": band5, "a1": a1, "a2": a2}
            ),
        },
    ).rename("SWI")

    return image.addBands(result)

In [13]:
# -----------------------------------------------------------------
#  4.2  Normalised Spectral Angle Index (NSAI)
# -----------------------------------------------------------------
def AI_vector(image, scale):
    """Compute the Normalised Spectral Angle Index (NSAI) using a
    vegetation-type-specific scale factor.

    Uses bands 1 (645 nm), 2 (859 nm), 5 (2130 nm).
    NSAI = acos(mm / dd) / π,  the normalised angle between the
    reflectance vector and the scaled reference vector.
    """
    R1, R2, R3 = 645.0, 859.0, 2130.0   # reference wavelengths (nm)
    r1_s, r2_s, r3_s = R1 * scale, R2 * scale, R3 * scale

    a1 = image.select("Nadir_Reflectance_Band1")   # ~645 nm
    a2 = image.select("Nadir_Reflectance_Band2")   # ~859 nm
    a3 = image.select("Nadir_Reflectance_Band5")   # ~2130 nm

    nsai = image.expression(
        "acos(mm / dd) / 3.141592653589793",
        {
            "dd": image.expression(
                "ee * cc",
                {
                    "ee": image.expression(
                        "sqrt((r3_s - r2_s) ** 2 + (a3 - a2) ** 2)",
                        {"r3_s": r3_s, "r2_s": r2_s, "a3": a3, "a2": a2},
                    ),
                    "cc": image.expression(
                        "sqrt((r1_s - r2_s) ** 2 + (a1 - a2) ** 2)",
                        {"r1_s": r1_s, "r2_s": r2_s, "a1": a1, "a2": a2},
                    ),
                },
            ),
            "mm": image.expression(
                "(r1_s - r2_s) * (r3_s - r2_s) + (a1 - a2) * (a3 - a2)",
                {
                    "r1_s": r1_s, "r2_s": r2_s, "r3_s": r3_s,
                    "a1": a1, "a2": a2, "a3": a3,
                },
            ),
        },
    ).rename("NSAI")

    return image.addBands(nsai)

In [14]:
# -----------------------------------------------------------------
#  4.3  Min–max normalisation to [−1, 1]
# -----------------------------------------------------------------
def normalize_band(image, band_name, x_min, x_max):
    """Normalise a single band to [−1, 1] using (x − min)/(max − min) × 2 − 1.

    The normalised band is added with the suffix '_Norm'.
    """
    band = image.select(band_name)
    x_min_img = ee.Image.constant(x_min)
    x_max_img = ee.Image.constant(x_max)
    norm = band.subtract(x_min_img).divide(x_max_img.subtract(x_min_img)) \
                .multiply(2).subtract(1)
    norm = norm.rename(f"{band_name}_Norm")
    return image.addBands(norm)

In [15]:
# -----------------------------------------------------------------
#  4.4  Model loading & weight extraction
#
#  Two model architectures coexist:
#    Grass models       → Layout A: fc1, fc2, fc3, fc4
#    Woody models (TH)  → Layout B: net.0, net.2, net.4, net.6
#
#  The code auto-detects the layout from the state_dict keys.
# -----------------------------------------------------------------

import zipfile
import pickle as _pickle


def _extract_state_dict(model_path):
    """Extract state_dict from a saved .pkl file."""

    # ---- Method 1: torch.load (handles zip + tensor references) ----
    try:
        obj = torch.load(
            model_path,
            map_location=torch.device("cpu"),
            weights_only=False,
        )
        if isinstance(obj, dict):
            return obj
        if hasattr(obj, "state_dict"):
            return obj.state_dict()
    except Exception:
        pass

    # ---- Method 2: native pickle via ZIP ----
    try:
        with zipfile.ZipFile(model_path, "r") as zf:
            with zf.open("archive/data.pkl") as f:
                model = _pickle.load(f)
        return model.state_dict()
    except (zipfile.BadZipFile, KeyError, Exception):
        pass

    # ---- Method 3: raw pickle (legacy format) ----
    try:
        with open(model_path, "rb") as f:
            model = _pickle.load(f)
        return model.state_dict()
    except Exception:
        pass

    raise RuntimeError(
        f"Could not load model from '{model_path}'. "
        f"Tried torch.load, zip+pickle, and raw pickle — all failed."
    )


def _detect_layout(state_dict):
    """Return True if the state_dict uses Sequential layout (net.0, net.2, …)."""
    return any(k.startswith("net.") for k in state_dict.keys())


def load_model(model_path, input_dim):
    """Load model weights and rebuild an MLP, auto-detecting the layout.

    Parameters
    ----------
    model_path : str
        Path to the .pkl file saved with torch.save(model, path).
    input_dim : int
        Number of input features (must match the saved model architecture).
    """
    state_dict = _extract_state_dict(model_path)
    use_seq = _detect_layout(state_dict)
    layout_name = "Sequential (net.0/net.2/…)" if use_seq else "fc1/fc2/fc3/fc4"
    print(f"  Loaded '{model_path}' → {layout_name}, input_dim={input_dim}")

    model = MLP(input_dim=input_dim, use_sequential=use_seq)
    model.load_state_dict(state_dict)
    model.eval()
    return model


def extract_weights(model):
    """Extract weights from MLP, handling both layouts.

    Layout A (fc1–fc4):   model.fc1.weight, model.fc2.weight, …
    Layout B (Sequential): model.net[0].weight, model.net[2].weight, …
    """
    if model.use_sequential:
        net = model.net
        return (
            net[0].weight.detach().cpu().numpy(),   # Linear  input_dim → 64
            net[0].bias.detach().cpu().numpy(),
            net[2].weight.detach().cpu().numpy(),   # Linear  64 → 128
            net[2].bias.detach().cpu().numpy(),
            net[4].weight.detach().cpu().numpy(),   # Linear  128 → 64
            net[4].bias.detach().cpu().numpy(),
            net[6].weight.detach().cpu().numpy(),   # Linear  64 → 2  (output)
            net[6].bias.detach().cpu().numpy(),
        )
    else:
        return (
            model.fc1.weight.detach().cpu().numpy(),
            model.fc1.bias.detach().cpu().numpy(),
            model.fc2.weight.detach().cpu().numpy(),
            model.fc2.bias.detach().cpu().numpy(),
            model.fc3.weight.detach().cpu().numpy(),
            model.fc3.bias.detach().cpu().numpy(),
            model.fc4.weight.detach().cpu().numpy(),
            model.fc4.bias.detach().cpu().numpy(),
        )


def apply_model(image, feature_band_names, weights):
    """Run the MLP forward pass inside Earth Engine using array operations.

    Weights order (matches extract_weights):
        (fc1_w, fc1_b, fc2_w, fc2_b, fc3_w, fc3_b, fc4_w, fc4_b)

    Architecture:
        fc1 → tanh → fc2 → tanh → fc3 → tanh → fc4 (no activation)

    Parameters
    ----------
    image : ee.Image
        Image with normalised input bands.
    feature_band_names : list[str]
        Ordered list of input band names (must match model training order).
    weights : tuple[ndarray × 8]
        (fc1_w, fc1_b, fc2_w, fc2_b, fc3_w, fc3_b, fc4_w, fc4_b).

    Returns
    -------
    ee.Image with bands 'output1' (EWT_norm) and 'output2' (Cm_norm).
    """
    fc1_w, fc1_b, fc2_w, fc2_b, fc3_w, fc3_b, fc4_w, fc4_b = weights

    # Convert NumPy weights to Earth Engine array images
    fc1_w_ee = ee.Image(ee.Array(fc1_w.tolist()))
    fc1_b_ee = ee.Image(ee.Array(fc1_b.tolist())).toArray(1)
    fc2_w_ee = ee.Image(ee.Array(fc2_w.tolist()))
    fc2_b_ee = ee.Image(ee.Array(fc2_b.tolist())).toArray(1)
    fc3_w_ee = ee.Image(ee.Array(fc3_w.tolist()))
    fc3_b_ee = ee.Image(ee.Array(fc3_b.tolist())).toArray(1)
    fc4_w_ee = ee.Image(ee.Array(fc4_w.tolist()))
    fc4_b_ee = ee.Image(ee.Array(fc4_b.tolist())).toArray(1)

    # Stack input bands into an array image of shape [n_features, 1]
    img_array = image.select(feature_band_names).toArray().toArray(1)

    # tanh activation expressed in Earth Engine primitives
    def _tanh(x):
        exp_x = x.exp()
        exp_neg_x = x.multiply(-1).exp()
        return exp_x.subtract(exp_neg_x).divide(exp_x.add(exp_neg_x))

    # Layer 1: fc1 → tanh
    t = fc1_w_ee.matrixMultiply(img_array).add(fc1_b_ee)
    t = _tanh(t)
    # Layer 2: fc2 → tanh
    t = fc2_w_ee.matrixMultiply(t).add(fc2_b_ee)
    t = _tanh(t)
    # Layer 3: fc3 → tanh
    t = fc3_w_ee.matrixMultiply(t).add(fc3_b_ee)
    t = _tanh(t)
    # Layer 4: fc4 — output (no activation)
    t = fc4_w_ee.matrixMultiply(t).add(fc4_b_ee)

    # Extract the 2 output neurons
    t1 = t.arrayGet([0, 0])
    t2 = t.arrayGet([1, 0])

    return ee.Image.cat([t1, t2]).rename(["output1", "output2"])

### 4.5 Core Retrieval Pipeline

The `_retrieval_pipeline` function is shared by all vegetation types. Each type
specifies its own `band_list` — woody types include `TreeHeight`, grass does not.
The model variant (with-LAI vs without-LAI) is selected based on LAI quality
**only**; tree height is always included when present in the band list.

In [16]:
# ---- Band lists (defined per vegetation type in VEGETATION_PARAMS) ----
# These are the REFERENCE lists; the actual lists live in the params dict below.
#
# Grass (no tree height):
#   with LAI    = 10 inputs: Band1–7 + LAI + NSAI + SWI
#   without LAI =  9 inputs: Band1–7 + NSAI + SWI
#
# Woody (with tree height):
#   with LAI    = 11 inputs: Band1–7 + LAI + NSAI + SWI + TreeHeight
#   without LAI = 10 inputs: Band1–7 + NSAI + SWI + TreeHeight

_BASE_REFL_BANDS = [
    "Nadir_Reflectance_Band3", "Nadir_Reflectance_Band4",
    "Nadir_Reflectance_Band1", "Nadir_Reflectance_Band2",
    "Nadir_Reflectance_Band5", "Nadir_Reflectance_Band6",
    "Nadir_Reflectance_Band7",
]

# Grass band lists (unchanged from baseline)
GRASS_BANDS_WITH_LAI    = _BASE_REFL_BANDS + ["Lai", "NSAI", "SWI"]
GRASS_BANDS_WITHOUT_LAI  = _BASE_REFL_BANDS + ["NSAI", "SWI"]

# Woody band lists (include TreeHeight as a fixed input)
WOODY_BANDS_WITH_LAI     = _BASE_REFL_BANDS + ["Lai", "NSAI", "TreeHeight", "SWI"]
WOODY_BANDS_WITHOUT_LAI  = _BASE_REFL_BANDS + ["NSAI", "SWI", "TreeHeight"]


def _retrieval_pipeline(image, x_min, x_max, model_path, nsai_scale,
                        ewt_ymax, ewt_ymin, cm_ymax, cm_ymin, band_list):
    """Shared FMC retrieval pipeline for one vegetation type.

    Steps:
        1. Compute SWI and NSAI.
        2. Normalise all input bands to [−1, 1] (including TreeHeight for woody types).
        3. Run the MLP → EWT_norm, Cm_norm.
        4. Inverse-normalise to physical EWT, Cm.
        5. Compute FMC = EWT / Cm × 100, clamped to [0, 400] as integer.

    Parameters
    ----------
    band_list : list[str]
        Ordered list of input band names for this vegetation type and model variant.
        Woody types include 'TreeHeight'; Grass does not.
        len(band_list) is passed as input_dim to the MLP constructor.
    """
    # Build a fresh MLP and load weights from the saved model file.
    # input_dim = len(band_list) ensures the architecture matches the data.
    model = load_model(model_path, input_dim=len(band_list))
    weights = extract_weights(model)

    # Spectral indices
    image = SACEWT(image)
    image = AI_vector(image, nsai_scale)

    # Normalise each input band
    for band_name in band_list:
        image = normalize_band(image, band_name, x_min[band_name], x_max[band_name])

    # MLP inference
    feature_bands = [f"{b}_Norm" for b in band_list]
    image = image.addBands(
        apply_model(image, feature_bands, weights).rename("EWT_Norm", "Cm_Norm")
    )

    # Inverse normalisation:  y = (y_norm + 1) / 2 × (ymax − ymin) + ymin
    denorm_expr = "max(min((a + 1) * (b - c) / 2 + c, b), c)"
    image = image.addBands(
        image.expression(denorm_expr, {
            "a": image.select("EWT_Norm"),
            "b": ewt_ymax, "c": ewt_ymin,
        }).rename("EWT")
    )
    image = image.addBands(
        image.expression(denorm_expr, {
            "a": image.select("Cm_Norm"),
            "b": cm_ymax, "c": cm_ymin,
        }).rename("Cm")
    )

    # FMC = EWT / Cm × 100, clamped to [0, 4] then scaled to integer [0, 400]
    fmc_expr = "int(max(min(a / b, 4), 0) * 100)"
    image = image.addBands(
        image.expression(fmc_expr, {
            "a": image.select("EWT"),
            "b": image.select("Cm"),
        }).rename("FMC")
    )

    return image.select("FMC", "reclassified", "EWT", "Cm")


def process_vegetation_type(image, x_min, x_max, model_path, nsai_scale,
                            ewt_ymax, ewt_ymin, cm_ymax, cm_ymin, band_list):
    """FMC retrieval WITH LAI as an input feature (plus TreeHeight if in band_list)."""
    return _retrieval_pipeline(
        image, x_min, x_max, model_path, nsai_scale,
        ewt_ymax, ewt_ymin, cm_ymax, cm_ymin, band_list,
    )


def process_vegetation_type_noLAI(image, x_min, x_max, model_path, nsai_scale,
                                  ewt_ymax, ewt_ymin, cm_ymax, cm_ymin, band_list):
    """FMC retrieval WITHOUT LAI (TreeHeight is still included if in band_list)."""
    return _retrieval_pipeline(
        image, x_min, x_max, model_path, nsai_scale,
        ewt_ymax, ewt_ymin, cm_ymax, cm_ymin, band_list,
    )

### 4.6 Vegetation-Type Parameters

Each vegetation type has its own:
- NSAI scale factor (calibrated per type)
- EWT / Cm min–max for inverse normalisation
- Per-band Xmin / Xmax for input normalisation (including TreeHeight for woody types)
- Band lists (`bands_with_lai`, `bands_without_lai`) — woody types include TreeHeight
- Paths to trained model weights

> **⚠️ Important:** The `TreeHeight` Xmin/Xmax values below are **placeholders**.
> Replace them with the actual min/max from your training data.
> Also update the `model_path` and `model_path_noLAI` entries to point to your
> newly trained models that include tree height as an input feature.

In [17]:
import os as _os

# TODO: Replace these placeholder TreeHeight Xmin/Xmax with actual training-data
# statistics. Typical canopy heights from GEDI rh98 range from 0–60 m globally;
# regional values may differ.


VEGETATION_PARAMS = {
    # ================================================================
    #  Type 2: Broadleaf Forest  (11 inputs with LAI, 10 without)
    # ================================================================
    "Broadleaf": {
        "type_num": 2,
        "uses_tree_height": True,
        "bands_with_lai":    WOODY_BANDS_WITH_LAI,
        "bands_without_lai": WOODY_BANDS_WITHOUT_LAI,
        "NSAI_scale": 0.001467544537815126,
        "EWT_Ymax": 0.054, "EWT_Ymin": 0.004,
        "Cm_Ymax":  0.038, "Cm_Ymin":  0.002,
        "Xmin": {
            "Nadir_Reflectance_Band1": 1.0000000e-06,
            "Nadir_Reflectance_Band2": 4.9116999e-02,
            "Nadir_Reflectance_Band3": 1.0000000e-06,
            "Nadir_Reflectance_Band4": 1.7000000e-05,
            "Nadir_Reflectance_Band5": 3.7638001e-02,
            "Nadir_Reflectance_Band6": 1.2173000e-02,
            "Nadir_Reflectance_Band7": 1.0000000e-06,
            "Lai":  5.0000000e-01,
            "NSAI": 6.5385914e-01,
            "SWI":  5.5437291e-01,
            "TreeHeight": 2.0000000e-01,   # TODO: calibrate
        },
        "Xmax": {
            "Nadir_Reflectance_Band1": 0.401019,
            "Nadir_Reflectance_Band2": 0.656336,
            "Nadir_Reflectance_Band3": 0.196209,
            "Nadir_Reflectance_Band4": 0.31088,
            "Nadir_Reflectance_Band5": 0.87319,
            "Nadir_Reflectance_Band6": 0.939135,
            "Nadir_Reflectance_Band7": 0.818187,
            "Lai":  8.0,
            "NSAI": 0.96155673,
            "SWI":  0.8356155,
            "TreeHeight": 41.2,   # TODO: calibrate
        },
        # TODO: update paths to models trained WITH tree height
        "model_path":         _os.path.join(MODEL_DIR, "Broadleaf-LAI-TH.pkl"),
        "model_path_noLAI":   _os.path.join(MODEL_DIR, "Broadleaf-noLAI-TH.pkl"),
    },
    # ================================================================
    #  Type 4: Shrub  (11 inputs with LAI, 10 without)
    # ================================================================
    "Shrub": {
        "type_num": 4,
        "uses_tree_height": True,
        "bands_with_lai":    WOODY_BANDS_WITH_LAI,
        "bands_without_lai": WOODY_BANDS_WITHOUT_LAI,
        "NSAI_scale": 0.0013277579831932772,
        "EWT_Ymax": 0.054, "EWT_Ymin": 0.004,
        "Cm_Ymax":  0.038, "Cm_Ymin":  0.002,
        "Xmin": {
            "Nadir_Reflectance_Band1": 0.060231,
            "Nadir_Reflectance_Band2": 0.137139,
            "Nadir_Reflectance_Band3": 0.016173,
            "Nadir_Reflectance_Band4": 0.040554,
            "Nadir_Reflectance_Band5": 0.154874,
            "Nadir_Reflectance_Band6": 0.140082,
            "Nadir_Reflectance_Band7": 0.126619,
            "Lai":  0.5,
            "NSAI":  0.7665583,
            "SWI":  0.7017228,
            "TreeHeight": 0.2,
        },
        "Xmax": {
            "Nadir_Reflectance_Band1": 0.386328,
            "Nadir_Reflectance_Band2": 0.633333,
            "Nadir_Reflectance_Band3": 0.194899,
            "Nadir_Reflectance_Band4": 0.305234,
            "Nadir_Reflectance_Band5": 0.850247,
            "Nadir_Reflectance_Band6": 0.902679,
            "Nadir_Reflectance_Band7": 0.785021,
            "Lai":  8.0,
            "NSAI": 0.9504289,
            "SWI":  0.8346782,
            "TreeHeight": 5.1,
        },
        # TODO: update paths to models trained WITH tree height
        "model_path":         _os.path.join(MODEL_DIR, "Shrub-LAI-TH.pkl"),
        "model_path_noLAI":   _os.path.join(MODEL_DIR, "Shrub-noLAI-TH.pkl"),
    },
    # ================================================================
    #  Type 3: Conifer Forest  (11 inputs with LAI, 10 without)
    # ================================================================
    "Conifer": {
        "type_num": 3,
        "uses_tree_height": True,
        "bands_with_lai":    WOODY_BANDS_WITH_LAI,
        "bands_without_lai": WOODY_BANDS_WITHOUT_LAI,
        "NSAI_scale": 0.0013479176470588233,
        "EWT_Ymax": 0.054, "EWT_Ymin": 0.004,
        "Cm_Ymax":  0.038, "Cm_Ymin":  0.002,
        "Xmin": {
            "Nadir_Reflectance_Band1": 1.0000000e-06,
            "Nadir_Reflectance_Band2": 3.3410002e-02,
            "Nadir_Reflectance_Band3": 1.0000000e-06,
            "Nadir_Reflectance_Band4": 4.9999999e-06,
            "Nadir_Reflectance_Band5": 2.5793999e-02,
            "Nadir_Reflectance_Band6": 1.0964000e-02,
            "Nadir_Reflectance_Band7": 1.3000000e-05,
            "Lai":  5.0000000e-01,
            "NSAI": 6.6554314e-01,
            "SWI":  5.5714786e-01,
            "TreeHeight": 2.0000000e-01,
        },
        "Xmax": {
            "Nadir_Reflectance_Band1": 0.378588,
            "Nadir_Reflectance_Band2": 0.617974,
            "Nadir_Reflectance_Band3": 0.195421,
            "Nadir_Reflectance_Band4": 0.291102,
            "Nadir_Reflectance_Band5": 0.802012,
            "Nadir_Reflectance_Band6": 0.81044,
            "Nadir_Reflectance_Band7": 0.701524,
            "Lai":  8.0,
            "NSAI": 0.9703919,
            "SWI":  0.8265793,
            "TreeHeight": 40.2,
        },
        # TODO: update paths to models trained WITH tree height
        "model_path":         _os.path.join(MODEL_DIR, "Conifer-LAI-TH.pkl"),
        "model_path_noLAI":   _os.path.join(MODEL_DIR, "Conifer-noLAI-TH.pkl"),
    },
    # ================================================================
    #  Type 1: Grass  (10 inputs with LAI, 9 without — NO tree height)
    # ================================================================
    "Grass": {
        "type_num": 1,
        "uses_tree_height": False,
        "bands_with_lai":    GRASS_BANDS_WITH_LAI,
        "bands_without_lai": GRASS_BANDS_WITHOUT_LAI,
        "NSAI_scale": 0.0003723398826046051,
        "EWT_Ymax": 0.097, "EWT_Ymin": 0.001,
        "Cm_Ymax":  0.053, "Cm_Ymin":  0.001,
        "Xmin": {
            "Nadir_Reflectance_Band1": 9.46770568e-08,
            "Nadir_Reflectance_Band2": 9.71318258e-02,
            "Nadir_Reflectance_Band3": 5.38758301e-09,
            "Nadir_Reflectance_Band4": 1.07576666e-03,
            "Nadir_Reflectance_Band5": 7.15844136e-02,
            "Nadir_Reflectance_Band6": 1.03212653e-02,
            "Nadir_Reflectance_Band7": 2.79864875e-08,
            "Lai":  1.00000000e-01,
            "NSAI": 2.93991000e-01,
            "SWI":  4.74718405e-01,
            # No TreeHeight for Grass
        },
        "Xmax": {
            "Nadir_Reflectance_Band1": 0.23133421,
            "Nadir_Reflectance_Band2": 0.55292475,
            "Nadir_Reflectance_Band3": 0.10643674,
            "Nadir_Reflectance_Band4": 0.16479264,
            "Nadir_Reflectance_Band5": 0.570653,
            "Nadir_Reflectance_Band6": 0.53004359,
            "Nadir_Reflectance_Band7": 0.50089978,
            "Lai":  8.1,
            "NSAI": 0.786934,
            "SWI":  0.84677844,
            # No TreeHeight for Grass
        },
        # Grass models unchanged from baseline (no tree height)
        "model_path":         _os.path.join(MODEL_DIR, "Grass-LAI.pkl"),
        "model_path_noLAI":   _os.path.join(MODEL_DIR, "Grass-noLAI.pkl"),
    },
}


# Lookup: vegetation type number → param dict key
VEG_TYPE_NUM_TO_KEY = {v["type_num"]: k for k, v in VEGETATION_PARAMS.items()}

print("Vegetation-type parameter summary:")
for key, p in VEGETATION_PARAMS.items():
    n_with = len(p["bands_with_lai"])
    n_without = len(p["bands_without_lai"])
    th_flag = "+TreeHeight" if p["uses_tree_height"] else "(no TreeHeight)"
    print(f"  {key:12s}  type={p['type_num']}  "
          f"inputs: {n_with}/{n_without} (with/without LAI)  {th_flag}")

Vegetation-type parameter summary:
  Broadleaf     type=2  inputs: 11/10 (with/without LAI)  +TreeHeight
  Shrub         type=4  inputs: 11/10 (with/without LAI)  +TreeHeight
  Conifer       type=3  inputs: 11/10 (with/without LAI)  +TreeHeight
  Grass         type=1  inputs: 10/9 (with/without LAI)  (no TreeHeight)


### 4.7 Vegetation-Type Dispatch

These functions select the correct model parameters and band list for a given
vegetation type, then run the retrieval pipeline. The band list is looked up
from `VEGETATION_PARAMS` — woody types automatically get `TreeHeight` included.

In [18]:
def _run_for_type(image, type_num, use_lai):
    """Run the retrieval pipeline for a single vegetation type.

    Parameters
    ----------
    type_num : int  — 1=Grass, 2=Broadleaf, 3=Conifer, 4=Shrub
    use_lai  : bool — if True, include LAI in the input features

    The correct band list (with or without TreeHeight) is automatically
    selected from VEGETATION_PARAMS based on the vegetation type.
    """
    key = VEG_TYPE_NUM_TO_KEY[type_num]
    params = VEGETATION_PARAMS[key]
    model_path = params["model_path"] if use_lai else params["model_path_noLAI"]
    band_list = params["bands_with_lai"] if use_lai else params["bands_without_lai"]
    pipeline = process_vegetation_type if use_lai else process_vegetation_type_noLAI
    return pipeline(
        image=image,
        x_min=params["Xmin"],
        x_max=params["Xmax"],
        model_path=model_path,
        nsai_scale=params["NSAI_scale"],
        ewt_ymax=params["EWT_Ymax"],
        ewt_ymin=params["EWT_Ymin"],
        cm_ymax=params["Cm_Ymax"],
        cm_ymin=params["Cm_Ymin"],
        band_list=band_list,
    )


def vegetation_type(image, type_num):
    """Run FMC retrieval WITH LAI for a given vegetation type number.

    Tree height is automatically included for woody types (2, 3, 4).
    """
    return _run_for_type(image, type_num, use_lai=True)


def vegetation_type_noLAI(image, type_num):
    """Run FMC retrieval WITHOUT LAI for a given vegetation type number.

    Tree height is automatically included for woody types (2, 3, 4).
    """
    return _run_for_type(image, type_num, use_lai=False)

## 5. Per-Pixel Model Selection & Compositing

In [19]:
def add_reclassified(image):
    """Attach the global reclassified land-cover band to an image."""
    return image.addBands(reclassified.rename("reclassified"))


def get_vegetation_data(image):
    """For each pixel, select the appropriate vegetation-type model (with or
    without LAI based on quality level) and return FMC, EWT, Cm, Quality_Level.

    Quality-level → model selection:
        Level 0 or 2 → LAI is available → use with-LAI model
        Level 1 or 3 → LAI is unavailable/poor → use without-LAI model

    Tree height is a **fixed** input: it is always included for woody types
    regardless of the LAI-based model variant.

    NOTE: This function computes all 8 model variants (4 veg types × 2 LAI
    variants) for every input image, then uses .where() to select per-pixel.
    This is computationally expensive. For large-scale production, consider
    splitting the image by vegetation type first (see optimisation note).
    """
    quality_level = image.select("Quality_Level")
    reclassified_band = image.select("reclassified")

    # Masks for model selection (tree height is ALWAYS included for woody types)
    has_lai_mask = quality_level.eq(0).Or(quality_level.eq(2))
    no_lai_mask  = quality_level.eq(1).Or(quality_level.eq(3))

    # Run all 8 model variants
    veg_results = {}
    for type_num in [1, 2, 3, 4]:
        with_lai = vegetation_type(image, type_num)
        without_lai = vegetation_type_noLAI(image, type_num)
        # Per-pixel: pick the with-LAI or without-LAI result based on quality
        combined = with_lai.where(no_lai_mask, without_lai)
        veg_results[type_num] = combined

    # Extract FMC, EWT, Cm for each vegetation type
    def _get_fmc_ewt_cm(veg_img):
        return (
            veg_img.select("FMC"),
            veg_img.select("EWT"),
            veg_img.select("Cm"),
        )

    fmc = {t: _get_fmc_ewt_cm(v)[0] for t, v in veg_results.items()}
    ewt = {t: _get_fmc_ewt_cm(v)[1] for t, v in veg_results.items()}
    cm  = {t: _get_fmc_ewt_cm(v)[2] for t, v in veg_results.items()}

    # Composite: select the correct vegetation-type result per pixel
    fmc_image = (
        fmc[1].where(reclassified_band.eq(1), fmc[1])
              .where(reclassified_band.eq(2), fmc[2])
              .where(reclassified_band.eq(3), fmc[3])
              .where(reclassified_band.eq(4), fmc[4])
              .rename("FMC")
    )
    ewt_image = (
        ewt[1].where(reclassified_band.eq(1), ewt[1])
              .where(reclassified_band.eq(2), ewt[2])
              .where(reclassified_band.eq(3), ewt[3])
              .where(reclassified_band.eq(4), ewt[4])
              .rename("EWT")
    )
    cm_image = (
        cm[1].where(reclassified_band.eq(1), cm[1])
              .where(reclassified_band.eq(2), cm[2])
              .where(reclassified_band.eq(3), cm[3])
              .where(reclassified_band.eq(4), cm[4])
              .rename("Cm")
    )

    # Mask non-vegetated pixels (reclassified == 255)
    veg_mask = reclassified_band.neq(255)
    fmc_image = fmc_image.updateMask(veg_mask)
    ewt_image = ewt_image.updateMask(veg_mask)
    cm_image  = cm_image.updateMask(veg_mask)
    quality_level = quality_level.updateMask(veg_mask)

    return (
        fmc_image
        .addBands([ewt_image, cm_image])
        .addBands(quality_level.rename("Quality_Level"))
        .updateMask(veg_mask)
    )

## 6. Run the Retrieval Pipeline

The processing chain now includes a step to attach the static GEDI tree height
band before model inference.

In [29]:
# -----------------------------------------------------------------
# Chain all processing steps via .map() over the image collection
#
#   mcd43a4 → add LAI → scale bands → QC → add land-cover
#           → add TreeHeight (NEW) → run retrieval models
# -----------------------------------------------------------------
print("Running retrieval pipeline with tree height...")

mcd43a4_add_lai      = mcd43a4.map(add_lai_to_mcd43a4)
mcd43a4_scaled       = mcd43a4_add_lai.map(process_image)
mcd43a4_qc           = mcd43a4_scaled.map(QCprocess_image)
mcd43a4_with_reclass = mcd43a4_qc.map(add_reclassified)
mcd43a4_with_height  = mcd43a4_with_reclass.map(add_tree_height)   # ← NEW step
fmc_result           = mcd43a4_with_height.map(get_vegetation_data)

# ---- Inspect the first result ----
first_image = fmc_result.first()
band_names = first_image.bandNames().getInfo()
print(f"Output bands: {band_names}")
# print(first_image.getInfo())   # uncomment for full metadata

Running retrieval pipeline with tree height...
  Loaded './Grass-LAI.pkl' → fc1/fc2/fc3/fc4, input_dim=10
  Loaded './Grass-noLAI.pkl' → fc1/fc2/fc3/fc4, input_dim=9
  Loaded './Broadleaf-LAI-TH.pkl' → Sequential (net.0/net.2/…), input_dim=11
  Loaded './Broadleaf-noLAI-TH.pkl' → Sequential (net.0/net.2/…), input_dim=10
  Loaded './Conifer-LAI-TH.pkl' → Sequential (net.0/net.2/…), input_dim=11
  Loaded './Conifer-noLAI-TH.pkl' → Sequential (net.0/net.2/…), input_dim=10
  Loaded './Shrub-LAI-TH.pkl' → Sequential (net.0/net.2/…), input_dim=11
  Loaded './Shrub-noLAI-TH.pkl' → Sequential (net.0/net.2/…), input_dim=10
  Loaded './Grass-LAI.pkl' → fc1/fc2/fc3/fc4, input_dim=10
  Loaded './Grass-noLAI.pkl' → fc1/fc2/fc3/fc4, input_dim=9
  Loaded './Broadleaf-LAI-TH.pkl' → Sequential (net.0/net.2/…), input_dim=11
  Loaded './Broadleaf-noLAI-TH.pkl' → Sequential (net.0/net.2/…), input_dim=10
  Loaded './Conifer-LAI-TH.pkl' → Sequential (net.0/net.2/…), input_dim=11
  Loaded './Conifer-noLAI-TH

## 7. Visualise (Optional)

Use `geemap` to quickly inspect the FMC map for one day.

In [30]:
# Example: display the FMC band for the first available image
Map = geemap.Map()
region = ee.Geometry.Rectangle([-118.8, 33.9, -118.2, 34.3])
fmc_vis = {
    "bands": ["FMC"],
    "min": 0,
    "max": 400,
    "palette": ["#ffffcc", "#a1dab4", "#41b6c4", "#2c7fb8", "#253494"],
}
Map.addLayer(first_image.clip(region), fmc_vis, "FMC (first image)")
Map.add_colorbar(fmc_vis, label="FMC (%)")
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

## 8. Export to Earth Engine Assets

In [31]:
# -----------------------------------------------------------------
# Export FMC + Quality_Level for each day in the configured range
# -----------------------------------------------------------------
GLOBAL_RECT = ee.Geometry.Rectangle([-180, -90, 180, 90], "EPSG:4326", False)

start_dt = datetime.strptime(EXPORT_START_DATE, "%Y-%m-%d")
end_dt   = datetime.strptime(EXPORT_END_DATE,   "%Y-%m-%d")

current_dt = start_dt
while current_dt < end_dt:
    date_str = current_dt.strftime("%Y-%m-%d")
    ee_start = ee.Date(date_str)
    ee_end   = ee_start.advance(1, "day")

    # Select FMC + Quality_Level for the day
    daily_image = (
        fmc_result
        .filterDate(ee_start, ee_end)
        .first()
        .select(["FMC", "Quality_Level"])
        .clip(GLOBAL_RECT)
    )

    asset_id = f"{GEE_ASSET_FOLDER}/{date_str}_FMC"

    task = ee.batch.Export.image.toAsset(
        image=daily_image,
        description=date_str,
        assetId=asset_id,
        crs="EPSG:4326",
        scale=EXPORT_SCALE_M,
        region=GLOBAL_RECT,
        maxPixels=EXPORT_MAX_PIXELS,
    )
    task.start()
    print(f"Export task for {date_str} started → {asset_id}")

    current_dt += timedelta(days=1)

print("All export tasks submitted.")

Export task for 2019-01-01 started → projects/ee-ahu1293659-10/assets/2019-01-01_FMC
All export tasks submitted.


### Tree Height Data

The static tree height layer is a **pre-processed derived product** created from
GEDI L4A V2.7 (`users/potapovpeter/GEDI_V27`, Potapov et al.) by:

1. Mosaicking the single 2019 image from the GEDI_V27 collection
2. Masking values > 60 m
3. Upscaling from native 0.00025° (~25–28 m) to 500 m using mean aggregation
4. Uploading to `projects/ee-ahu1293659-11/assets/forest_height_2019_global`

**Resulting asset properties:**

| Property | Value |
|----------|-------|
| Band | `b1` (renamed to `TreeHeight` in code) |
| Data type | Unsigned int8 |
| Value range | 0–60 |
| CRS | EPSG:4326 |
| Nominal scale | ~500 m |

> **⚠️  IMPORTANT — Original data source:**
> This is a convenience asset derived from the original GEDI L4A V2.7 data.
> Users of this notebook MUST:
> 1. Access the original data at `users/potapovpeter/GEDI_V27`
> 2. Cite the original GEDI data (Potapov et al.)
> 3. See `DATA.md` in the repository for full provenance and processing steps